
# Stadtbach – MILP-Optimierung (Pyomo) *robustes Notebook*

Dieses Notebook lädt die Eingabedaten (Wärmebedarf, Strompreise, Abwärmeströme), bereitet sie robust auf, baut ein **MILP**-Modell mit **exogenen COP-Parametern** (Lookup/Interpolation), löst es (Gurobi bevorzugt, sonst CBC/GLPK) und erstellt aussagekräftige Plots & KPIs.

**Hinweise**
- Bitte Pfade in **Abschnitt 1** anpassen (Dateien lokal ablegen).
- Wenn Gurobi nicht verfügbar ist, versucht das Notebook automatisch `cbc` oder `glpk`.
- Alle Auswertungen laufen erst **nach erfolgreichem Solve** (ansonsten werden KPIs/Plots übersprungen).


In [1]:
# 1) Imports & Versionen
# ============================
import os, sys, math, glob, bisect, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Pyomo
import pyomo.environ as pyo
from pyomo.environ import (
    ConcreteModel, Var, Param, RangeSet, Set, Constraint, Objective, Expression,
    NonNegativeReals, PositiveReals, Binary, Reals, minimize, quicksum, value as pyo_val
)
from pyomo.opt import SolverFactory

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
try:
    import pyomo
    print("Pyomo:", pyomo.version.__version__)
except Exception:
    print("Pyomo: OK (Version unbekannt)")

# Zeitschrittbreite [h]: 1.0=stündlich, 0.5=30min, 0.25=15min, ...
DT_H = 1.0   # <- auf 0.25 setzen, wenn du 15-Minuten-Daten hast


# ============================
# 2) Pfade / Konfiguration
# ============================
WAERME_XLSX        = "Waermebedarf.xlsx"
PREISE_CSV_PATTERN = "Data/GUI_ENERGY_PRICES_202212312300-202312312300.csv"
ABWAERME_CSV       = "Abwärmestroeme.csv"
BENCH_HP_XLSX      = "hp_waste_heat_2023_20GWh_minimal.xlsx"   # Prüfstands-Daten

# Solver-Präferenzen
PREFERRED_SOLVERS = ["gurobi"]

# Modeling flags
INCLUDE_GRIDCOST_IN_ENERGY = False
USE_RAMPS = True

# ============================
# 3) Loader & Utils
# ============================
def load_waermebedarf_xlsx(path, jahr="2022"):
    if not os.path.exists(path):
        print(f"[WARN] Datei nicht gefunden: {path}")
        return pd.Series([], dtype=float)
    df = pd.read_excel(path)
    df.columns = df.columns.astype(str).str.strip()
    needed = ["Datum", "Zeit", jahr]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        print(f"[WARN] Spalten fehlen in {path}: {missing}. Verfügbare: {df.columns.tolist()}")
        return pd.Series([], dtype=float)
    s = pd.to_numeric(df[jahr], errors="coerce").fillna(0.0).astype(float)
    s.loc[s < 0] = 0.0
    return s

def load_strompreise_csv(pattern):
    matches = glob.glob(pattern)
    if not matches:
        print(f"[WARN] Keine Datei gefunden für Pattern: {pattern}")
        return pd.Series([], dtype=float)
    path = sorted(matches)[0]
    df = pd.read_csv(path)
    if "Sequence" in df.columns:
        df = df[df["Sequence"] == "Sequence 1"]
    keep = [c for c in ["MTU (CET/CEST)", "Day-ahead (EUR/MWh)"] if c in df.columns]
    if len(keep) < 2:
        print(f"[WARN] Erwartete Spalten nicht gefunden in {path}. Spalten: {df.columns.tolist()}")
        return pd.Series([], dtype=float)
    df = df[keep].copy()
    df["MTU_Start"] = df["MTU (CET/CEST)"].astype(str).str.split(" - ", expand=True)[0]
    df["MTU_Start"] = pd.to_datetime(df["MTU_Start"], format="%d/%m/%Y %H:%M:%S", dayfirst=True, errors="coerce")
    df = df.set_index("MTU_Start")
    s = pd.to_numeric(df["Day-ahead (EUR/MWh)"], errors="coerce").resample("H").mean()
    s = s.fillna(method="ffill").fillna(method="bfill")
    return s

def load_abwaermestroeme_csv(path):
    if not os.path.exists(path):
        print(f"[WARN] Datei nicht gefunden: {path}")
        return pd.DataFrame()
    last_err, df = None, None
    for enc in ("latin-1", "cp1252", "utf-8-sig"):
        for hdr in (3, 0):
            try:
                df = pd.read_csv(path, sep=";", decimal=",", encoding=enc, header=hdr)
                break
            except Exception as e:
                last_err = e
                df = None
        if df is not None:
            break
    if df is None:
        print(f"[WARN] CSV konnte nicht gelesen werden: {last_err}")
        return pd.DataFrame()

    df.columns = (
        df.columns.astype(str)
          .str.replace("\u00a0", " ", regex=False)
          .str.strip()
          .str.replace(r"\s+", " ", regex=True)
    )

    date_col = None
    for c in df.columns:
        s = df[c].astype(str)
        if s.str.contains(r"\d{2}\.\d{2}\.\d{4}\s+\d{2}:\d{2}", regex=True).any():
            date_col = c
            break
    if date_col is None:
        candidates = [c for c in df.columns if re.search(r"datum", c, re.I)]
        if candidates:
            date_col = candidates[0]
    if date_col is None:
        print(f"[WARN] Keine Datumsspalte gefunden. Spalten: {df.columns.tolist()}")
        return pd.DataFrame()
    df = df.rename(columns={date_col: "Datum"})
    df["Datum"] = pd.to_datetime(df["Datum"], dayfirst=True, errors="coerce")
    if df["Datum"].isna().all():
        print("[WARN] Datum konnte nicht geparst werden.")
        return pd.DataFrame()

    # WRG3 T.1 -> WRG3 Q fix
    if "WRG3 T.1" in df.columns and "WRG3 Q" not in df.columns:
        df = df.rename(columns={"WRG3 T.1": "WRG3 Q"})

    keep = ["Datum", "WRG1 T", "WRG1 Q", "WRG2 T", "WRG2 Q", "WRG3 T", "WRG3 Q"]
    present = [c for c in keep if c in df.columns]
    if "Datum" not in present:
        present = ["Datum"] + present
    df = df[present].copy()
    num_cols = [c for c in df.columns if c != "Datum"]
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df[df["Datum"].dt.year == 2023]
    if df.empty:
        print("[WARN] Keine Daten für Jahr 2023 gefunden – prüfe CSV.")
        return pd.DataFrame()

    df = df.set_index("Datum")
    agg = {}
    for c in df.columns:
        if c.endswith(" T"):
            agg[c] = "mean"
        elif c.endswith(" Q"):
            agg[c] = "sum"
    hourly = df.resample("H").agg(agg)

    fill = {}
    for c in hourly.columns:
        fill[c] = 293.15 if c.endswith(" T") else 0.0
    hourly = hourly.fillna(fill)
    hourly = hourly.reset_index()
    hourly["t"] = range(1, len(hourly)+1)
    return hourly

def load_bench_hp_excel(path):
    """
    Erwartet Spalten: ['timestamp','Maschinenleistung in MW','is_test','T_source_HP'].
    Retour: (Q_MW list, T_src_K list, is_test list)
    """
    if not os.path.exists(path):
        print(f"[WARN] Bench-HP Datei nicht gefunden: {path}")
        return [], [], []
    df = pd.read_excel(path, sheet_name="data", parse_dates=["timestamp"])
    req = ["timestamp", "Maschinenleistung in MW", "is_test", "T_source_HP"]
    miss = [c for c in req if c not in df.columns]
    if miss:
        print(f"[WARN] Spalten fehlen in {path}: {miss}")
        return [], [], []
    df = df.sort_values("timestamp").reset_index(drop=True)
    q   = pd.to_numeric(df["Maschinenleistung in MW"], errors="coerce").fillna(0.0).clip(lower=0.0)
    t_C = pd.to_numeric(df["T_source_HP"], errors="coerce").fillna(35.0)
    flg = pd.to_numeric(df["is_test"], errors="coerce").fillna(0).astype(int).clip(0,1)
    return q.tolist(), (t_C + 273.15).tolist(), flg.tolist()

def clean_series(arr, default):
    out = []
    for v in arr:
        try:
            vv = float(v)
            if not np.isfinite(vv):
                vv = default
        except Exception:
            vv = default
        out.append(vv)
    return out

def align_len(*lists):
    L = min(len(x) for x in lists if len(x) > 0) if lists else 0
    if L == 0:
        return [list() for _ in lists], 0
    return [x[:L] for x in lists], L

# ============================
# 4) Daten laden & harmonisieren
# ============================
waerme_series  = load_waermebedarf_xlsx(WAERME_XLSX, jahr="2022")
preise_series  = load_strompreise_csv(PREISE_CSV_PATTERN)
hourly         = load_abwaermestroeme_csv(ABWAERME_CSV)
bench_Q_list, bench_TK_list, bench_flag_list = load_bench_hp_excel(BENCH_HP_XLSX)

print("Wärmebedarf Werte:", len(waerme_series))
print("Strompreis Werte:", len(preise_series))
print("Abwärme Stunden:", len(hourly))
print("Bench HP Werte:", len(bench_Q_list))

if isinstance(hourly, pd.DataFrame) and not hourly.empty:
    WRG1_T_list = hourly.get("WRG1 T", pd.Series([293.15]*len(hourly))).astype(float).tolist()
    WRG2_T_list = hourly.get("WRG2 T", pd.Series([293.15]*len(hourly))).astype(float).tolist()
    WRG3_T_list = hourly.get("WRG3 T", pd.Series([293.15]*len(hourly))).astype(float).tolist()
    WRG1Q_list  = hourly.get("WRG1 Q", pd.Series([0.0]*len(hourly))).astype(float).tolist()
    WRG2Q_list  = hourly.get("WRG2 Q", pd.Series([0.0]*len(hourly))).astype(float).tolist()
    WRG3Q_list  = hourly.get("WRG3 Q", pd.Series([0.0]*len(hourly))).astype(float).tolist()
else:
    WRG1_T_list, WRG2_T_list, WRG3_T_list = [], [], []
    WRG1Q_list,  WRG2Q_list,  WRG3Q_list  = [], [], []

waerme = clean_series(waerme_series, 0.0)
preise = clean_series(preise_series, 0.0)
WRG1_T_list = clean_series(WRG1_T_list, 293.15)
WRG2_T_list = clean_series(WRG2_T_list, 293.15)
WRG3_T_list = clean_series(WRG3_T_list, 293.15)
WRG1Q_list  = clean_series(WRG1Q_list, 0.0)
WRG2Q_list  = clean_series(WRG2Q_list, 0.0)
WRG3Q_list  = clean_series(WRG3Q_list, 0.0)
bench_TK_list = clean_series(bench_TK_list, 308.15)  # 35°C -> Kelvin
bench_Q_list  = clean_series(bench_Q_list, 0.0)
bench_flag_list = [int(1 if v else 0) for v in bench_flag_list]

([preise, waerme,
  WRG1_T_list, WRG2_T_list, WRG3_T_list,
  WRG1Q_list,  WRG2Q_list,  WRG3Q_list,
  bench_TK_list, bench_Q_list, bench_flag_list], T) = align_len(
    preise, waerme,
    WRG1_T_list, WRG2_T_list, WRG3_T_list,
    WRG1Q_list,  WRG2Q_list,  WRG3Q_list,
    bench_TK_list, bench_Q_list, bench_flag_list
)
print("Harmonisierte Länge T =", T)

# ============================
# 5) COP-Logik & Hilfsfunktionen
# ============================
Tsink_out = 363.15
Tsink_in  = 343.15
deltaTpp  = 5.0
eta       = 0.75
FQ        = 0.10

def lmtd(Th, Tc):
    return (Th - Tc) / np.log(Th / Tc)

LMTD_sink = lmtd(Tsink_out, Tsink_in)

Tsourcein_vals = np.linspace(303.15, 353.15, 6)   # 30..80°C K
deltaT_vals    = np.array([20, 30, 40, 50])
COP_MIN_HP = 1.01
COP_MAX_HP = 12.0
_records = []
for Tsourcein in Tsourcein_vals:
    for dT in deltaT_vals:
        Tsourceout = Tsourcein - dT
        if Tsourceout <= 0 or Tsourceout >= Tsourcein:
            continue
        LMTD_source = lmtd(Tsourcein, Tsourceout)
        mdts = 0.2*(Tsink_out - Tsourceout + 2*deltaTpp) + 0.2*(Tsink_out - Tsink_in) + 0.016
        qww  = 0.0014*(Tsink_out - Tsourceout + 2*deltaTpp) - 0.0015*(Tsink_out - Tsink_in) + 0.039
        A = LMTD_sink / (LMTD_sink - LMTD_source + 1e-9)
        B = (1 + (mdts + deltaTpp)/LMTD_sink) / (1 + (mdts + 0.5*dT + 2*deltaTpp)/(LMTD_sink - LMTD_source + 1e-9))
        COP = A * B * eta * (1 - qww) + 1 - eta - FQ
        COP = float(np.clip(COP if np.isfinite(COP) else 3.0, 0.5, 12.0))
        _records.append({"Tsourcein": round(Tsourcein,2), "Tsourceout": round(Tsourceout,2), "COP": round(COP,4)})

cop_lookup_df = pd.DataFrame(_records).sort_values(["Tsourcein", "Tsourceout"])

from collections import defaultdict
cop_piecewise_rules = defaultdict(list)
for _, r in cop_lookup_df.iterrows():
    cop_piecewise_rules[r["Tsourcein"]].append((r["Tsourceout"], r["COP"] ))
cop_piecewise_rules = {k: sorted(v, key=lambda x: x[0]) for k, v in cop_piecewise_rules.items()}

_all_x = sorted({x for pts in cop_piecewise_rules.values() for (x, _) in pts})
x_min, x_max = (_all_x[0], _all_x[-1]) if _all_x else (0.0, 1.0)
T_grid = sorted(cop_piecewise_rules.keys())

def _interp_on_curve(pts, x):
    if x <= pts[0][0]:  return pts[0][1]
    if x >= pts[-1][0]: return pts[-1][1]
    for (x0,y0),(x1,y1) in zip(pts[:-1], pts[1:]):
        if x0 <= x <= x1:
            return y0 + (y1-y0) * (x - x0) / (x1 - x0)
    return pts[-1][1]

def bilinear_interp_from_lookup(Tsrc_in, x, clamp_x=True):
    if not T_grid:
        return 3.0
    if clamp_x:
        x = max(x_min, min(x_max, x))
    j = bisect.bisect_left(T_grid, Tsrc_in)
    if j == 0:
        return _interp_on_curve(cop_piecewise_rules[T_grid[0]], x)
    if j == len(T_grid):
        return _interp_on_curve(cop_piecewise_rules[T_grid[-1]], x)
    t0, t1 = T_grid[j-1], T_grid[j]
    y0 = _interp_on_curve(cop_piecewise_rules[t0], x)
    y1 = _interp_on_curve(cop_piecewise_rules[t1], x)
    if not np.isfinite(y0): y0 = 3.0
    if not np.isfinite(y1): y1 = 3.0
    w  = (Tsrc_in - t0) / (t1 - t0)
    val = y0*(1-w) + y1*w
    return float(np.clip(val if np.isfinite(val) else 3.0, 0.5, 12.0))

def safe_Tout(Tin, dT=20.0, default_T=293.15):
    Tin = float(Tin) if np.isfinite(Tin) else default_T
    if Tin <= dT + 1e-6:
        Tin = default_T
    Tout = Tin - dT
    return max(1.0, Tout)

# (Nur 1x definieren – bereinigt)
def safe_cop(Tin, Tout, COP_MIN=COP_MIN_HP, COP_MAX=COP_MAX_HP, COP_FALLBACK=3.0):
    try:
        Tin_v, Tout_v = float(Tin), float(Tout)
    except:
        return COP_FALLBACK
    x = max(x_min, min(x_max, Tout_v)) if x_min < x_max else Tout_v
    val = bilinear_interp_from_lookup(Tin_v, x) if T_grid else COP_FALLBACK
    if (not np.isfinite(val)) or (val <= 0):
        val = COP_FALLBACK
    return float(np.clip(val, COP_MIN, COP_MAX))



Python: 3.6.3 |Anaconda, Inc.| (default, Nov  8 2017, 15:10:56) [MSC v.1900 64 bit (AMD64)]
NumPy: 1.18.5
Pandas: 1.1.5
Pyomo: 5.7.3
Wärmebedarf Werte: 8760
Strompreis Werte: 8760
Abwärme Stunden: 8760
Bench HP Werte: 8760
Harmonisierte Länge T = 8760


In [2]:
# ============================
# 6) Model Build (lean & fast)
# ============================

if len(preise) == 0 or len(waerme) == 0:
    print("[STOP] Keine harmonisierten Daten – bitte Pfade/Daten prüfen.")
    MODEL_BUILT = False
else:
    # ---------------------------
    # Anlagen-Schalter (Szenarien)
    # ---------------------------
    ENABLE_CONFIG = {
        "HP1": True, "HP2": True, "HP3": True, "HP4": True,
        "HKW": True, "GTOST": True, "P2H": True,
        "BMHKW": True, "HWS": True, "HWW": True, "AVA": True,
        "STORAGE": True,
    }

    # optional: globale Flags aus früher übernehmen (sonst Defaults)
    USE_RAMPS = globals().get("USE_RAMPS", True)

    model = ConcreteModel(name="Stadtbach")

    # --- Sets
    model.t = RangeSet(1, T)

    STEPS_PER_DAY = int(round(24 / DT_H))   # statt 24
    n_days = max(1, T // STEPS_PER_DAY)
    model.d = RangeSet(1, n_days)

    def _t_day_init(m):
        return [(d, tt)
                for d in m.d
                for tt in range((d - 1) * STEPS_PER_DAY + 1, min(d * STEPS_PER_DAY, T) + 1)]
    model.t_day = Set(dimen=2, initialize=_t_day_init)

    # --- Ökonomie & Preise
    model.strompreis   = Param(model.t, initialize={i: preise[i-1] for i in range(1, T+1)}, default=0.0)
    model.waermebedarf = Param(model.t, initialize={i: waerme[i-1]  for i in range(1, T+1)}, default=0.0)

    model.Leistungspreis      = Param(initialize=127240)   # €/MW_el·a
    model.Gritcost            = Param(initialize=61.6)     # €/MWh_el
    model.Installationskosten = Param(initialize=20000)    # €/Anlage
    model.Gaspreis            = Param(initialize=58.6)     # €/MWh_th
    model.Abfallpreis         = Param(initialize=10)       # €/MWh_th
    model.Biomassepreis       = Param(initialize=20)       # €/MWh_th
 
        # Einspeise-Floor (mutable -> Laufzeit-Tuning ohne Rebuild)
    model.einspeisepreis = Param(initialize=0.0, within=NonNegativeReals, mutable=True)

    # Verkaufspreis-Tuning (Baseline-Formel als Expression + ReLU-Floor)
    model.SELL_HAIRCUT = Param(initialize=0.05, mutable=True)
    model.SELL_SPREAD  = Param(initialize=5.0,  mutable=True)
    model.SELL_FEE     = Param(initialize=5.0,  mutable=True)
    model.SELL_PREMIUM = Param(initialize=0.0,  mutable=True)

    model.sell_base = Expression(
    model.t,
    rule=lambda m, t: (1.0 - m.SELL_HAIRCUT) * m.strompreis[t]
                      + m.SELL_PREMIUM - m.SELL_FEE - m.SELL_SPREAD
    )
        # --- Einspeise-Floor (Param) ist schon da:
    # model.einspeisepreis = Param(initialize=..., mutable=True)

    # Effektiver Verkaufspreis als Param (= max(sell_base, einspeisepreis))
    model.sell_price_eff = pyo.Param(
        model.t,
        initialize=lambda m, t: max(float(pyo_val(m.sell_base[t])),
                                    float(pyo_val(m.einspeisepreis))),
        mutable=True
    )

    # Recompute-Helfer (wenn Haircut/Spread/Fee/Premium/Floor geändert werden)
    def refresh_sell_price_eff(m):
        floor = float(pyo_val(m.einspeisepreis))
        for t in m.t:
            base_t = float(pyo_val(m.sell_base[t]))
            m.sell_price_eff[t].set_value(max(base_t, floor))

    # Funktion am Modell „anheften“, damit _solve_once sie findet
    model.refresh_sell_price_eff = refresh_sell_price_eff


    # --------------------------
    # Enable-Parameter (mutable)
    # --------------------------
    model.EN_HP1 = Param(initialize=int(ENABLE_CONFIG["HP1"]), within=NonNegativeReals, mutable=True)
    model.EN_HP2 = Param(initialize=int(ENABLE_CONFIG["HP2"]), within=NonNegativeReals, mutable=True)
    model.EN_HP3 = Param(initialize=int(ENABLE_CONFIG["HP3"]), within=NonNegativeReals, mutable=True)
    model.EN_HP4 = Param(initialize=int(ENABLE_CONFIG["HP4"]), within=NonNegativeReals, mutable=True)

    model.EN_HKW     = Param(initialize=int(ENABLE_CONFIG["HKW"]),     within=NonNegativeReals, mutable=True)
    model.EN_GTOST   = Param(initialize=int(ENABLE_CONFIG["GTOST"]),   within=NonNegativeReals, mutable=True)
    model.EN_P2H     = Param(initialize=int(ENABLE_CONFIG["P2H"]),     within=NonNegativeReals, mutable=True)
    model.EN_BMHKW   = Param(initialize=int(ENABLE_CONFIG["BMHKW"]),   within=NonNegativeReals, mutable=True)
    model.EN_HWS     = Param(initialize=int(ENABLE_CONFIG["HWS"]),     within=NonNegativeReals, mutable=True)
    model.EN_HWW     = Param(initialize=int(ENABLE_CONFIG["HWW"]),     within=NonNegativeReals, mutable=True)
    model.EN_AVA     = Param(initialize=int(ENABLE_CONFIG["AVA"]),     within=NonNegativeReals, mutable=True)
    model.EN_STORAGE = Param(initialize=int(ENABLE_CONFIG["STORAGE"]), within=NonNegativeReals, mutable=True)

    # --- Speicher (E & P)
    model.storage_eff_charge    = Param(initialize=0.95)
    model.storage_eff_discharge = Param(initialize=0.95)
    #model.storage_loss          = Param(initialize=0.9999)   # stündlicher SoC-Faktor
    model.storage_c_rate        = Param(initialize=0.25)     # [1/h] max. P/E
    model.CAPEXspeicher       = Param(initialize=5160, within=PositiveReals, mutable=True)  # €/MWh_th (Energie)
    model.Lebensdauerspeicher = Param(initialize=20)   # a
    STO_E_MAX = 50000.0  # MWh_th
    STO_E_MIN = 50.0  # MWh_th
    STO_P_MAX = 2000.0   # MW_th
    model.storage_capacity        = Var(bounds=(0, STO_E_MAX))
    model.storage_power           = Var(bounds=(0, STO_P_MAX))
    model.storage_capacity_active = Var(domain=Binary)
    model.storage_level     = Var(model.t, domain=NonNegativeReals)
    model.storage_charge    = Var(model.t, domain=NonNegativeReals)   # MW_th
    model.storage_discharge = Var(model.t, domain=NonNegativeReals)   # MW_th
    model.SOC_init          = Param(initialize=0.0, within=NonNegativeReals, mutable=True)
    # Exklusivität Laden/Entladen (Binary). Optional eliminierbar – siehe Hinweise unten.
    model.sto_mode          = Var(model.t, domain=Binary)  # 1 = Laden, 0 = Entladen
    model.sto_mode_gate = pyo.Constraint(model.t, rule=lambda m,t: m.sto_mode[t] <= m.storage_capacity_active)
    model.storage_power_rate = Constraint(expr=model.storage_power <= model.storage_c_rate * model.storage_capacity)
    # Gates
    model.sto_gate_charge    = Constraint(model.t, rule=lambda m, t: m.storage_charge[t]    <= m.storage_power * m.EN_STORAGE * m.sto_mode[t])
    model.sto_gate_discharge = Constraint(model.t, rule=lambda m, t: m.storage_discharge[t] <= m.storage_power * m.EN_STORAGE * (1 - m.sto_mode[t]))
    model.soc_cap            = Constraint(model.t, rule=lambda m, t: m.storage_level[t]     <= m.storage_capacity * m.EN_STORAGE)
    # Aktivschaltung Speicher
    model.storage_linkE_hi   = Constraint(expr=model.storage_capacity <= STO_E_MAX * 
                                          model.storage_capacity_active * model.EN_STORAGE)
    model.storage_linkE_lo   = Constraint(expr=model.storage_capacity >= STO_E_MIN * 
                                          model.storage_capacity_active * model.EN_STORAGE)
    model.storage_active_gate= Constraint(expr=model.storage_capacity_active <= model.EN_STORAGE)
     # vorher: model.storage_loss = Param(initialize=0.9999)  # stündlicher SoC-Faktor
    model.storage_loss_hour = Param(initialize=0.9999, mutable=True)  # 1h-Faktor
    # besser als Param: Expression, dann ist ein Tuning von storage_loss_hour sofort wirksam
    model.storage_loss_step = pyo.Expression(rule=lambda m: m.storage_loss_hour ** DT_H)
    def storage_state_rule(m, t):
        prev = (m.storage_level[t-1] * m.storage_loss_step) if t > m.t.first() else m.SOC_init
        # charge/discharge sind MW_th → * DT_H für MWh_th
        return prev + m.storage_charge[t] * DT_H - m.storage_discharge[t] * DT_H == m.storage_level[t]
    model.storage_state = Constraint(model.t, rule=storage_state_rule)
    model.soc_terminal = pyo.Constraint(expr = model.storage_level[model.t.last()] >= model.SOC_init)
    # --- WPs / Design-Parameter
    model.LebensdauerHP          = Param(initialize=15)
    model.CapexHP                = Param(initialize=600000, within=PositiveReals, mutable=True)
    model.HPstarts               = Param(initialize=10)
    model.HP_min_load_fraction   = Param(initialize=0.30)   # realistische Mindestlast
    model.HP_ramp_fraction_per_h = Param(initialize=0.50)   # optional, nur wenn USE_RAMPS

    # --- WRG-Parameter (1..4 inkl. Bench)
    model.WRG1T = Param(model.t, initialize={i: WRG1_T_list[i-1] for i in range(1, T+1)}, within=Reals, default=293.15, mutable=True)
    model.WRG2T = Param(model.t, initialize={i: WRG2_T_list[i-1] for i in range(1, T+1)}, within=Reals, default=293.15, mutable=True)
    model.WRG3T = Param(model.t, initialize={i: WRG3_T_list[i-1] for i in range(1, T+1)}, within=Reals, default=293.15, mutable=True)
    model.WRG1Q = Param(model.t, initialize={i: WRG1Q_list[i-1]  for i in range(1, T+1)}, within=NonNegativeReals, default=0.0, mutable=True)
    model.WRG2Q = Param(model.t, initialize={i: WRG2Q_list[i-1]  for i in range(1, T+1)}, within=NonNegativeReals, default=0.0, mutable=True)
    model.WRG3Q = Param(model.t, initialize={i: WRG3Q_list[i-1]  for i in range(1, T+1)}, within=NonNegativeReals, default=0.0, mutable=True)

    _fallback_TK = 308.15
    _fallback_Q  = 0.0
    WRG4T_init = {i: (bench_TK_list[i-1] if len(bench_TK_list) >= i else _fallback_TK) for i in range(1, T+1)}
    WRG4Q_init = {i: (bench_Q_list[i-1]  if len(bench_Q_list)  >= i else _fallback_Q ) for i in range(1, T+1)}
    BENCH_IS_TEST_init = {i: (bench_flag_list[i-1] if len(bench_flag_list) >= i else 0) for i in range(1, T+1)}

    model.WRG4T         = Param(model.t, initialize=WRG4T_init,         within=Reals,            default=293.15, mutable=True)
    model.WRG4Q         = Param(model.t, initialize=WRG4Q_init,         within=NonNegativeReals, default=0.0,    mutable=True)


    # COPs (exogen, robust)
    model.Tsourceout1 = Param(model.t, initialize={i: safe_Tout(WRG1_T_list[i-1]) for i in range(1, T+1)}, within=PositiveReals)
    model.Tsourceout2 = Param(model.t, initialize={i: safe_Tout(WRG2_T_list[i-1]) for i in range(1, T+1)}, within=PositiveReals)
    model.Tsourceout3 = Param(model.t, initialize={i: safe_Tout(WRG3_T_list[i-1]) for i in range(1, T+1)}, within=PositiveReals)

    model.COP1 = Param(model.t, initialize={i: safe_cop(WRG1_T_list[i-1], safe_Tout(WRG1_T_list[i-1])) for i in range(1, T+1)}, within=PositiveReals)
    model.COP2 = Param(model.t, initialize={i: safe_cop(WRG2_T_list[i-1], safe_Tout(WRG2_T_list[i-1])) for i in range(1, T+1)}, within=PositiveReals)
    model.COP3 = Param(model.t, initialize={i: safe_cop(WRG3_T_list[i-1], safe_Tout(WRG3_T_list[i-1])) for i in range(1, T+1)}, within=PositiveReals)

    Tsourceout4_init = {i: safe_Tout(WRG4T_init[i]) for i in range(1, T+1)}
    COP4_init        = {i: safe_cop (WRG4T_init[i], Tsourceout4_init[i]) for i in range(1, T+1)}
    model.Tsourceout4 = Param(model.t, initialize=Tsourceout4_init, within=PositiveReals)
    model.COP4        = Param(model.t, initialize=COP4_init,        within=PositiveReals)

    Tsrc_default = 23.0 + 273.15
    X_default    = Tsrc_default - 20.0
    model.COPdefault = Param(initialize=safe_cop(Tsrc_default, X_default), within=PositiveReals)

    # --- Konventionelle Einheiten
    model.HKW_th_eff   = Param(initialize=0.743)
    model.HKW_el_eff   = Param(initialize=0.177)
    model.GTOST_th_eff = Param(initialize=0.466)
    model.GTOST_el_eff = Param(initialize=0.36)
    model.P2H_eff      = Param(initialize=0.99)
    model.BMHKW_th_eff = Param(initialize=0.485)
    model.BMHKW_el_eff = Param(initialize=0.177)
    model.HWS_th_eff   = Param(initialize=0.936)
    model.HWW_th_eff   = Param(initialize=0.924)
    model.AVA_th_eff   = Param(initialize=1.0)

    # --- Variablen (konventionell)
    model.HKW_Power   = Var(model.t, domain=NonNegativeReals, bounds=(0, 75))
    model.GTOST_Power = Var(model.t, domain=NonNegativeReals, bounds=(0, 41.3))
    model.P2H_Power   = Var(model.t, domain=NonNegativeReals, bounds=(0, 10))
    model.BMHKW_Power = Var(model.t, domain=NonNegativeReals, bounds=(0, 15))
    model.HWS_Power   = Var(model.t, domain=NonNegativeReals, bounds=(0, 45))
    model.HWW_Power   = Var(model.t, domain=NonNegativeReals, bounds=(0, 45))
    model.AVA_Power   = Var(model.t, domain=NonNegativeReals, bounds=(0, 45))

    # --- HP Nennleistungen & On-Variablen (1..4)
    HP_MAX = 100.0   # eng setzen → bessere LP-Relaxation
    HP_MIN = 1.0     # falls Mindestgröße gewünscht: 1.0 setzen

    model.HPNenn1 = Var(bounds=(0, HP_MAX))
    model.HPNenn2 = Var(bounds=(0, HP_MAX))
    model.HPNenn3 = Var(bounds=(0, HP_MAX))
    model.HPNenn4 = Var(bounds=(0, HP_MAX))

    model.HPNenn_active1 = Var(domain=Binary)
    model.HPNenn_active2 = Var(domain=Binary)
    model.HPNenn_active3 = Var(domain=Binary)
    model.HPNenn_active4 = Var(domain=Binary)

    model.onHP1 = Var(model.t, domain=Binary)
    model.onHP2 = Var(model.t, domain=Binary)
    model.onHP3 = Var(model.t, domain=Binary)
    model.onHP4 = Var(model.t, domain=Binary)

    # --- HP Wärmeflüsse
    model.Qwrg1 = Var(model.t, domain=NonNegativeReals)
    model.Qwrg2 = Var(model.t, domain=NonNegativeReals)
    model.Qwrg3 = Var(model.t, domain=NonNegativeReals)
    model.Qwrg4 = Var(model.t, domain=NonNegativeReals)
    model.Qdef1 = Var(model.t, domain=NonNegativeReals)
    model.Qdef2 = Var(model.t, domain=NonNegativeReals)
    model.Qdef3 = Var(model.t, domain=NonNegativeReals)
    model.Qdef4 = Var(model.t, domain=NonNegativeReals)
    model.Qhp1  = Var(model.t, domain=NonNegativeReals)
    model.Qhp2  = Var(model.t, domain=NonNegativeReals)
    model.Qhp3  = Var(model.t, domain=NonNegativeReals)
    model.Qhp4  = Var(model.t, domain=NonNegativeReals)

    # --- Quellen-Caps
    model.wrg1_cap = Constraint(model.t, rule=lambda m, t: m.Qwrg1[t] <= m.WRG1Q[t])
    model.wrg2_cap = Constraint(model.t, rule=lambda m, t: m.Qwrg2[t] <= m.WRG2Q[t])
    model.wrg3_cap = Constraint(model.t, rule=lambda m, t: m.Qwrg3[t] <= m.WRG3Q[t])
    model.wrg4_cap = Constraint(model.t, rule=lambda m, t: m.Qwrg4[t] <= m.WRG4Q[t])

    # --- HP-Bilanzen
    model.hp1_balance = Constraint(model.t, rule=lambda m, t: m.Qhp1[t] == m.Qwrg1[t] + m.Qdef1[t])
    model.hp2_balance = Constraint(model.t, rule=lambda m, t: m.Qhp2[t] == m.Qwrg2[t] + m.Qdef2[t])
    model.hp3_balance = Constraint(model.t, rule=lambda m, t: m.Qhp3[t] == m.Qwrg3[t] + m.Qdef3[t])
    model.hp4_balance = Constraint(model.t, rule=lambda m, t: m.Qhp4[t] == m.Qwrg4[t] + m.Qdef4[t])

    # --- Kapazitäten & Mindestlast (pro HP)
    def _hp_caps(m, Qhp, HPNenn, on):
        min_fr = m.HP_min_load_fraction
        return [
            pyo.Constraint(m.t, rule=lambda m, t, Qhp=Qhp, HPNenn=HPNenn: Qhp[t] <= HPNenn),
            pyo.Constraint(m.t, rule=lambda m, t, Qhp=Qhp, on=on:       Qhp[t] <= HP_MAX * on[t]),
            pyo.Constraint(m.t, rule=lambda m, t, Qhp=Qhp, HPNenn=HPNenn, on=on:
                Qhp[t] >= min_fr * HPNenn - (1 - on[t]) * min_fr * HP_MAX)
        ]
    model.hp1_cap1, model.hp1_cap2, model.hp1_min = _hp_caps(model, model.Qhp1, model.HPNenn1, model.onHP1)
    model.hp2_cap1, model.hp2_cap2, model.hp2_min = _hp_caps(model, model.Qhp2, model.HPNenn2, model.onHP2)
    model.hp3_cap1, model.hp3_cap2, model.hp3_min = _hp_caps(model, model.Qhp3, model.HPNenn3, model.onHP3)
    model.hp4_cap1, model.hp4_cap2, model.hp4_min = _hp_caps(model, model.Qhp4, model.HPNenn4, model.onHP4)

    # --- Installation-Links (mit Enable-Gates)
    model.on_impl_1 = Constraint(model.t, rule=lambda m, t: m.onHP1[t] <= m.HPNenn_active1)
    model.on_impl_2 = Constraint(model.t, rule=lambda m, t: m.onHP2[t] <= m.HPNenn_active2)
    model.on_impl_3 = Constraint(model.t, rule=lambda m, t: m.onHP3[t] <= m.HPNenn_active3)
    model.on_impl_4 = Constraint(model.t, rule=lambda m, t: m.onHP4[t] <= m.HPNenn_active4)

    model.hp1_link_hi = Constraint(expr=model.HPNenn1 <= HP_MAX * model.HPNenn_active1 * model.EN_HP1)
    model.hp2_link_hi = Constraint(expr=model.HPNenn2 <= HP_MAX * model.HPNenn_active2 * model.EN_HP2)
    model.hp3_link_hi = Constraint(expr=model.HPNenn3 <= HP_MAX * model.HPNenn_active3 * model.EN_HP3)
    model.hp4_link_hi = Constraint(expr=model.HPNenn4 <= HP_MAX * model.HPNenn_active4 * model.EN_HP4)

    model.hp1_link_lo = Constraint(expr=model.HPNenn1 >= HP_MIN * model.HPNenn_active1 * model.EN_HP1)
    model.hp2_link_lo = Constraint(expr=model.HPNenn2 >= HP_MIN * model.HPNenn_active2 * model.EN_HP2)
    model.hp3_link_lo = Constraint(expr=model.HPNenn3 >= HP_MIN * model.HPNenn_active3 * model.EN_HP3)
    model.hp4_link_lo = Constraint(expr=model.HPNenn4 >= HP_MIN * model.HPNenn_active4 * model.EN_HP4)

    model.hp1_active_gate = Constraint(expr=model.HPNenn_active1 <= model.EN_HP1)
    model.hp2_active_gate = Constraint(expr=model.HPNenn_active2 <= model.EN_HP2)
    model.hp3_active_gate = Constraint(expr=model.HPNenn_active3 <= model.EN_HP3)
    model.hp4_active_gate = Constraint(expr=model.HPNenn_active4 <= model.EN_HP4)

    model.hp1_enable_on = Constraint(model.t, rule=lambda m, t: m.onHP1[t] <= m.EN_HP1)
    model.hp2_enable_on = Constraint(model.t, rule=lambda m, t: m.onHP2[t] <= m.EN_HP2)
    model.hp3_enable_on = Constraint(model.t, rule=lambda m, t: m.onHP3[t] <= m.EN_HP3)
    model.hp4_enable_on = Constraint(model.t, rule=lambda m, t: m.onHP4[t] <= m.EN_HP4)


    # --- Rampen (optional)
    if USE_RAMPS:
        def _make_ramps(name_prefix, Qhp, HPNenn):
            fr = model.HP_ramp_fraction_per_h
            setattr(
                model,
                f"{name_prefix}_RampUp",
                pyo.Constraint(
                    model.t,
                    rule=lambda m, t, Qhp=Qhp, HPNenn=HPNenn, fr=fr:
                        pyo.Constraint.Skip if t == m.t.first() else Qhp[t] - Qhp[t-1] <= fr * HPNenn * DT_H
                )
            )
            setattr(
                model,
                f"{name_prefix}_RampDown",
                pyo.Constraint(
                    model.t,
                    rule=lambda m, t, Qhp=Qhp, HPNenn=HPNenn, fr=fr:
                        pyo.Constraint.Skip if t == m.t.first() else Qhp[t-1] - Qhp[t] <= fr * HPNenn * DT_H
                )
            )
        _make_ramps("HP1", model.Qhp1, model.HPNenn1)
        _make_ramps("HP2", model.Qhp2, model.HPNenn2)
        _make_ramps("HP3", model.Qhp3, model.HPNenn3)
        _make_ramps("HP4", model.Qhp4, model.HPNenn4)

    # --- Starts / Tag (HP1..HP4)
    model.start1 = Var(model.t, domain=Binary)
    model.start2 = Var(model.t, domain=Binary)
    model.start3 = Var(model.t, domain=Binary)
    model.start4 = Var(model.t, domain=Binary)

    model.start1a = Constraint(model.t, rule=lambda m, t: (m.start1[t] == m.onHP1[t]) if t == m.t.first() else pyo.Constraint.Skip)
    model.start1b = Constraint(model.t, rule=lambda m, t: (m.start1[t] >= m.onHP1[t] - m.onHP1[t-1]) if t > m.t.first() else pyo.Constraint.Skip)
    model.start1c = Constraint(model.t, rule=lambda m, t: (m.start1[t] <= m.onHP1[t]) if t > m.t.first() else pyo.Constraint.Skip)
    model.start1d = Constraint(model.t, rule=lambda m, t: (m.start1[t] <= 1 - m.onHP1[t-1]) if t > m.t.first() else pyo.Constraint.Skip)

    model.start2a = Constraint(model.t, rule=lambda m, t: (m.start2[t] == m.onHP2[t]) if t == m.t.first() else pyo.Constraint.Skip)
    model.start2b = Constraint(model.t, rule=lambda m, t: (m.start2[t] >= m.onHP2[t] - m.onHP2[t-1]) if t > m.t.first() else pyo.Constraint.Skip)
    model.start2c = Constraint(model.t, rule=lambda m, t: (m.start2[t] <= m.onHP2[t]) if t > m.t.first() else pyo.Constraint.Skip)
    model.start2d = Constraint(model.t, rule=lambda m, t: (m.start2[t] <= 1 - m.onHP2[t-1]) if t > m.t.first() else pyo.Constraint.Skip)

    model.start3a = Constraint(model.t, rule=lambda m, t: (m.start3[t] == m.onHP3[t]) if t == m.t.first() else pyo.Constraint.Skip)
    model.start3b = Constraint(model.t, rule=lambda m, t: (m.start3[t] >= m.onHP3[t] - m.onHP3[t-1]) if t > m.t.first() else pyo.Constraint.Skip)
    model.start3c = Constraint(model.t, rule=lambda m, t: (m.start3[t] <= m.onHP3[t]) if t > m.t.first() else pyo.Constraint.Skip)
    model.start3d = Constraint(model.t, rule=lambda m, t: (m.start3[t] <= 1 - m.onHP3[t-1]) if t > m.t.first() else pyo.Constraint.Skip)

    model.start4a = Constraint(model.t, rule=lambda m, t: (m.start4[t] == m.onHP4[t]) if t == m.t.first() else pyo.Constraint.Skip)
    model.start4b = Constraint(model.t, rule=lambda m, t: (m.start4[t] >= m.onHP4[t] - m.onHP4[t-1]) if t > m.t.first() else pyo.Constraint.Skip)
    model.start4c = Constraint(model.t, rule=lambda m, t: (m.start4[t] <= m.onHP4[t]) if t > m.t.first() else pyo.Constraint.Skip)
    model.start4d = Constraint(model.t, rule=lambda m, t: (m.start4[t] <= 1 - m.onHP4[t-1]) if t > m.t.first() else pyo.Constraint.Skip)

    model.daily_starts1 = Constraint(model.d, rule=lambda m, d: pyo.quicksum(m.start1[t] for (dd, t) in m.t_day if dd == d) <= m.HPstarts)
    model.daily_starts2 = Constraint(model.d, rule=lambda m, d: pyo.quicksum(m.start2[t] for (dd, t) in m.t_day if dd == d) <= m.HPstarts)
    model.daily_starts3 = Constraint(model.d, rule=lambda m, d: pyo.quicksum(m.start3[t] for (dd, t) in m.t_day if dd == d) <= m.HPstarts)
    model.daily_starts4 = Constraint(model.d, rule=lambda m, d: pyo.quicksum(m.start4[t] for (dd, t) in m.t_day if dd == d) <= m.HPstarts)

    # --- Wärme-Erzeugung als Expression + 1 Bilanz (>= Demand)
    model.total_heat = pyo.Expression(
        model.t,
        rule=lambda m, t: (
            m.Qhp1[t] + m.Qhp2[t] + m.Qhp3[t] + m.Qhp4[t]
          + m.HKW_Power[t]   * m.HKW_th_eff   * m.EN_HKW
          + m.GTOST_Power[t] * m.GTOST_th_eff * m.EN_GTOST
          + m.P2H_Power[t]   * m.P2H_eff      * m.EN_P2H
          + m.BMHKW_Power[t] * m.BMHKW_th_eff * m.EN_BMHKW
          + m.HWS_Power[t]   * m.HWS_th_eff   * m.EN_HWS
          + m.HWW_Power[t]   * m.HWW_th_eff   * m.EN_HWW
          + m.AVA_Power[t]   * m.AVA_th_eff   * m.EN_AVA
        )
    )

    model.waerme_demand = pyo.Constraint(
        model.t,
        rule=lambda m, t:
            m.total_heat[t]
          - m.storage_charge[t]    / m.storage_eff_charge
          + m.storage_discharge[t] * m.storage_eff_discharge
          >= m.waermebedarf[t]
    )

    # --- Elektrizitätsbilanz
    P_FLOW_CAP = 300.0  # eng setzen, wenn realistisch
    model.P_buy  = Var(model.t, domain=NonNegativeReals, bounds=(0, P_FLOW_CAP))
    model.P_sell = Var(model.t, domain=NonNegativeReals, bounds=(0, P_FLOW_CAP))

    model.e_balance = Constraint(
        model.t,
        rule=lambda m, t:
            (
                m.HKW_Power[t]*m.HKW_el_eff     * m.EN_HKW
              + m.GTOST_Power[t]*m.GTOST_el_eff * m.EN_GTOST
              + m.BMHKW_Power[t]*m.BMHKW_el_eff * m.EN_BMHKW
              + m.P_buy[t]
            )
            - (
                m.Qwrg1[t]/m.COP1[t] + m.Qdef1[t]/m.COPdefault
              + m.Qwrg2[t]/m.COP2[t] + m.Qdef2[t]/m.COPdefault
              + m.Qwrg3[t]/m.COP3[t] + m.Qdef3[t]/m.COPdefault
              + m.Qwrg4[t]/m.COP4[t] + m.Qdef4[t]/m.COPdefault
              + m.P2H_Power[t] * m.EN_P2H
              + m.P_sell[t]
            )
            == 0
    )

    # --- Gates für konventionelle Einheiten (Bounds × Enable)
    model.hkw_gate   = Constraint(model.t, rule=lambda m, t: m.HKW_Power[t]   <= 75   * m.EN_HKW)
    model.gtost_gate = Constraint(model.t, rule=lambda m, t: m.GTOST_Power[t] <= 41.3 * m.EN_GTOST)
    model.p2h_gate   = Constraint(model.t, rule=lambda m, t: m.P2H_Power[t]   <= 10   * m.EN_P2H)
    model.bmhkw_gate = Constraint(model.t, rule=lambda m, t: m.BMHKW_Power[t] <= 15   * m.EN_BMHKW)
    model.hws_gate   = Constraint(model.t, rule=lambda m, t: m.HWS_Power[t]   <= 45   * m.EN_HWS)
    model.hww_gate   = Constraint(model.t, rule=lambda m, t: m.HWW_Power[t]   <= 45   * m.EN_HWW)
    model.ava_gate   = Constraint(model.t, rule=lambda m, t: m.AVA_Power[t]   <= 45   * m.EN_AVA)

    # --- Gasverbrauch als Expression (statt Var + Gleichung)
    model.Gasverbrauch = pyo.Expression(
        model.t,
        rule=lambda m, t: (
            m.HKW_Power[t]   * m.EN_HKW
          + m.GTOST_Power[t] * m.EN_GTOST
          + m.HWS_Power[t]   * m.EN_HWS
          + m.HWW_Power[t]   * m.EN_HWW
        )
    )

    # --- Spitzenlast / Netz
    model.max_stromverbrauch = Var(domain=NonNegativeReals)
    model.max_stromverbrauch_constraint = Constraint(model.t, rule=lambda m, t: m.max_stromverbrauch >= m.P_buy[t])

    # --- Preise
    
    model.buy_price = pyo.Expression(model.t, rule=lambda m, t: m.strompreis[t] + m.Gritcost)

    # Kein gleichzeitiges Kaufen/Verkaufen (Binary). Optional eliminierbar – siehe Hinweise.
    model.M_GRID    = Param(initialize=P_FLOW_CAP)
    model.grid_mode = Var(model.t, domain=Binary)  # 1 = kaufen, 0 = verkaufen
    model.buy_gate  = Constraint(model.t, rule=lambda m, t: m.P_buy[t]  <= m.M_GRID * m.grid_mode[t])
    model.sell_gate = Constraint(model.t, rule=lambda m, t: m.P_sell[t] <= m.M_GRID * (1 - m.grid_mode[t]))

    # --- Objective
    HOURS_PER_YEAR = 8760.0
    model.year_frac = Param(initialize=float(T) * DT_H / HOURS_PER_YEAR)

    model.total_cost = Objective(
        expr=(
              pyo.quicksum(DT_H * model.P_buy[t]  * model.buy_price[t]       for t in model.t)
            - pyo.quicksum(DT_H * model.P_sell[t] * model.sell_price_eff[t]  for t in model.t)
            + pyo.quicksum(DT_H * model.Gasverbrauch[t] * model.Gaspreis     for t in model.t)
            + pyo.quicksum(DT_H * model.AVA_Power[t]    * model.Abfallpreis  for t in model.t)
            + pyo.quicksum(DT_H * model.BMHKW_Power[t]  * model.Biomassepreis for t in model.t)
            + (model.HPNenn1 + model.HPNenn2 + model.HPNenn3 + model.HPNenn4) * model.CapexHP / model.LebensdauerHP * model.year_frac
            + model.CAPEXspeicher * model.storage_capacity / model.Lebensdauerspeicher * model.year_frac
            + model.Installationskosten * (model.HPNenn_active1 + model.HPNenn_active2 + model.HPNenn_active3 + model.HPNenn_active4) / model.LebensdauerHP * model.year_frac
            + model.Installationskosten * (model.storage_capacity_active) / model.Lebensdauerspeicher * model.year_frac
            + model.Leistungspreis * model.max_stromverbrauch * model.year_frac
        ),
        sense=pyo.minimize
    )

    MODEL_BUILT = True
    print("Model built with T =", T)


Model built with T = 8760


In [6]:
# ==== Timeseries-Export (mit ExcelWriter-Unterstützung für Version B) ====
import pandas as pd
import numpy as np
from pyomo.environ import value as pyo_value
from typing import Optional


def _series(model, comp):
    """Indexed (t) Param/Var/Expression -> pandas.Series"""
    idx = _series._idx
    return pd.Series([float(pyo_value(comp[t])) for t in model.t], index=idx)

def _scalar_series(x):
    """Skalar -> Series über Zeitachse dupliziert"""
    idx = _series._idx
    return pd.Series([float(pyo_value(x))]*len(idx), index=idx)

def _make_index(model, DT_H, idx=None, start_ts=None, tz="Europe/Berlin"):
    if idx is not None:
        return pd.Index(idx, name="t")
    if start_ts is not None:
        steps = len(list(model.t))
        return pd.date_range(pd.Timestamp(start_ts, tz=tz),
                             periods=steps,
                             freq=pd.Timedelta(hours=float(DT_H)),
                             name="time")
    return pd.RangeIndex(1, len(list(model.t))+1, name="t")

def export_timeseries(
    model, DT_H,
    idx: Optional[pd.Index] = None,
    start_ts: Optional[str] = None,
    tz: str = "Europe/Berlin",
    out_xlsx: str = "dispatch_timeseries.xlsx",
    out_csv: Optional[str] = None,
    writer: Optional[pd.ExcelWriter] = None,
    sheet_prefix: Optional[str] = None,
):

    """
    Schreibt mehrere Sheets:
      - Dispatch_MW: Leistungen (MW_th / MW_el), SoC (MWh_th)
      - Prices: Strom-/Kosten-Preise (€/MWh)
      - Binaries: on-Variablen, Modi
      - Checks: Bilanzen/Residuals, Bedarf
      - Energies_MWh: obige Leistungen * DT_H
    Wenn 'writer' gesetzt ist, wird in dieses Workbook geschrieben (Version B).
    """
    _series._idx = _make_index(model, DT_H, idx=idx, start_ts=start_ts, tz=tz)
    dt = float(DT_H)
    val = pyo_value

    dispatch = pd.DataFrame({
        "Q_HP1_MWth": _series(model, model.Qhp1),
        "Q_HP2_MWth": _series(model, model.Qhp2),
        "Q_HP3_MWth": _series(model, model.Qhp3),
        "Q_HP4_MWth": _series(model, model.Qhp4),

        "Qwrg1_MWth": _series(model, model.Qwrg1),
        "Qdef1_MWth": _series(model, model.Qdef1),
        "Qwrg2_MWth": _series(model, model.Qwrg2),
        "Qdef2_MWth": _series(model, model.Qdef2),
        "Qwrg3_MWth": _series(model, model.Qwrg3),
        "Qdef3_MWth": _series(model, model.Qdef3),
        "Qwrg4_MWth": _series(model, model.Qwrg4),
        "Qdef4_MWth": _series(model, model.Qdef4),

        "HKW_Fuel_MW":   _series(model, model.HKW_Power),
        "GTOST_Fuel_MW": _series(model, model.GTOST_Power),
        "BMHKW_Fuel_MW": _series(model, model.BMHKW_Power),
        "HWS_Fuel_MW":   _series(model, model.HWS_Power),
        "HWW_Fuel_MW":   _series(model, model.HWW_Power),
        "AVA_Fuel_MW":   _series(model, model.AVA_Power),

        "Sto_Charge_MWth":    _series(model, model.storage_charge),
        "Sto_Discharge_MWth": _series(model, model.storage_discharge),
        "Sto_SoC_MWhth":      _series(model, model.storage_level),

        "P_buy_MWel":  _series(model, model.P_buy),
        "P_sell_MWel": _series(model, model.P_sell),
    })

    dispatch["HKW_Heat_MWth"]    = dispatch["HKW_Fuel_MW"]   * float(val(model.HKW_th_eff))
    dispatch["HKW_Power_MWel"]   = dispatch["HKW_Fuel_MW"]   * float(val(model.HKW_el_eff))
    dispatch["GTOST_Heat_MWth"]  = dispatch["GTOST_Fuel_MW"] * float(val(model.GTOST_th_eff))
    dispatch["GTOST_Power_MWel"] = dispatch["GTOST_Fuel_MW"] * float(val(model.GTOST_el_eff))
    dispatch["BMHKW_Heat_MWth"]  = dispatch["BMHKW_Fuel_MW"] * float(val(model.BMHKW_th_eff))
    dispatch["BMHKW_Power_MWel"] = dispatch["BMHKW_Fuel_MW"] * float(val(model.BMHKW_el_eff))
    dispatch["HWS_Heat_MWth"]    = dispatch["HWS_Fuel_MW"]   * float(val(model.HWS_th_eff))
    dispatch["HWW_Heat_MWth"]    = dispatch["HWW_Fuel_MW"]   * float(val(model.HWW_th_eff))
    dispatch["AVA_Heat_MWth"]    = dispatch["AVA_Fuel_MW"]   * float(val(model.AVA_th_eff))

    total_heat = pd.Series([float(val(model.total_heat[t])) for t in model.t], index=_series._idx)
    dispatch["TotalHeat_MWth"] = total_heat
    dispatch["Heat_to_Demand_MWth"] = (
        total_heat
        - dispatch["Sto_Charge_MWth"]    / float(val(model.storage_eff_charge))
        + dispatch["Sto_Discharge_MWth"] * float(val(model.storage_eff_discharge))
    )

    hp_el = (
        _series(model, model.Qwrg1) / _series(model, model.COP1) + _series(model, model.Qdef1) / float(val(model.COPdefault)) +
        _series(model, model.Qwrg2) / _series(model, model.COP2) + _series(model, model.Qdef2) / float(val(model.COPdefault)) +
        _series(model, model.Qwrg3) / _series(model, model.COP3) + _series(model, model.Qdef3) / float(val(model.COPdefault)) +
        _series(model, model.Qwrg4) / _series(model, model.COP4) + _series(model, model.Qdef4) / float(val(model.COPdefault))
    )
    dispatch["HP_El_MWel"]  = hp_el
    dispatch["P2H_El_MWel"] = _series(model, model.P2H_Power)

    prices = pd.DataFrame({
        "Strompreis_EUR_MWh":     _series(model, model.strompreis),
        "BuyPrice_EUR_MWh":       pd.Series([float(val(model.buy_price[t])) for t in model.t], index=_series._idx),
        "SellBase_EUR_MWh":       pd.Series([float(val(model.sell_base[t])) for t in model.t], index=_series._idx),
        "SellEff_EUR_MWh":        pd.Series([float(val(model.sell_price_eff[t])) for t in model.t], index=_series._idx),
        "EinspeiseFloor_EUR_MWh": _scalar_series(model.einspeisepreis),
        "Gaspreis_EUR_MWhth":     _scalar_series(model.Gaspreis),
        "Biomasse_EUR_MWhth":     _scalar_series(model.Biomassepreis),
        "Abfall_EUR_MWhth":       _scalar_series(model.Abfallpreis),
    })

    bins = pd.DataFrame({
        "onHP1": _series(model, model.onHP1),
        "onHP2": _series(model, model.onHP2),
        "onHP3": _series(model, model.onHP3),
        "onHP4": _series(model, model.onHP4),
        "grid_mode_buy1_sell0": _series(model, model.grid_mode),
        "sto_mode_charge1":     _series(model, model.sto_mode),
    })

    waermebedarf = _series(model, model.waermebedarf)
    ebal_residual = (
        dispatch["HKW_Power_MWel"] + dispatch["GTOST_Power_MWel"] + dispatch["BMHKW_Power_MWel"] + dispatch["P_buy_MWel"]
        - (dispatch["HP_El_MWel"] + dispatch["P2H_El_MWel"] + dispatch["P_sell_MWel"])
    )
    checks = pd.DataFrame({
        "Waermebedarf_MWth": waermebedarf,
        "HeatSlack_MWth": (dispatch["Heat_to_Demand_MWth"] - waermebedarf),
        "E_Balance_Residual_MWel": ebal_residual,
        "Gasverbrauch_Fuel_MW": _series(model, model.Gasverbrauch),
        "max_Stromverbrauch_MWel": _scalar_series(model.max_stromverbrauch),
    })

    energies = dispatch.filter(regex=r"_MW(th|el)|_MW$").multiply(dt)
    energies.columns = [c.replace("_MW", "_MWh") for c in energies.columns]

    # --- Export: entweder in bestehendes Workbook (Version B) oder in Datei
    if writer is not None:
        sp = (sheet_prefix or "").strip()
        def sn(name):
            base = f"{sp}_{name}" if sp else name
            base = base.replace("/", "_").replace("\\", "_")
            return base[:31]  # Excel Limit
        dispatch.to_excel(writer, sheet_name=sn("Dispatch_MW"))
        prices.to_excel(writer,   sheet_name=sn("Prices"))
        bins.to_excel(writer,     sheet_name=sn("Binaries"))
        checks.to_excel(writer,   sheet_name=sn("Checks"))
        energies.to_excel(writer, sheet_name=sn("Energies_MWh"))
    elif out_xlsx:
        with pd.ExcelWriter(out_xlsx) as xl:
            dispatch.to_excel(xl, sheet_name="Dispatch_MW")
            prices.to_excel(xl,   sheet_name="Prices")
            bins.to_excel(xl,     sheet_name="Binaries")
            checks.to_excel(xl,   sheet_name="Checks")
            energies.to_excel(xl, sheet_name="Energies_MWh")

    if out_csv:
        dispatch.to_csv(out_csv, index=True)

    return {"dispatch": dispatch, "prices": prices, "binaries": bins, "checks": checks, "energies": energies}


# ============================
# 7) Optimierung über Parameterbereiche (fix & clean)
# ============================

# Beispiel: 200k..800k (200k-Schritt) und 1k..7k (2k-Schritt)
capex_hp_values = np.arange(200000, 800001, 200000)
capex_speicher_values = np.arange(1000, 7001, 2000)

results = []

# Solver einmal bauen + konfigurieren
solver = SolverFactory('gurobi')
solver.options.update({
    'TimeLimit': 3600,      # 30 min
    'MIPGap': 0.1,         # 10%
    'LogToConsole': 1,
    'Threads': 30,
    'Heuristics': 0.5,
    'Cuts': 2,
    'Presolve': 2,
    'Method': 1,
})

dt = DT_H
val = pyo.value  # Alias

ok_tc = {
    pyo.TerminationCondition.optimal,
    pyo.TerminationCondition.locallyOptimal,
    pyo.TerminationCondition.feasible,
    pyo.TerminationCondition.maxTimeLimit,
}

# >>> Version B: eine Sammel-Excel für alle Szenarien
with pd.ExcelWriter("timeseries_all_szenarien.xlsx") as writer:
    for capex_hp in capex_hp_values:
        model.CapexHP.set_value(float(capex_hp))
        for capex_speicher in capex_speicher_values:
            model.CAPEXspeicher.set_value(float(capex_speicher))

            # Falls SELL_* oder einspeisepreis verändert werden:
            # model.refresh_sell_price_eff(model)

            result = solver.solve(model, tee=True, load_solutions=True)

            if (result.solver.status in (pyo.SolverStatus.ok, pyo.SolverStatus.warning) and
                result.solver.termination_condition in ok_tc):

                storage_cap = val(model.storage_capacity)
                total_cost  = val(model.total_cost)

                # Energiemengen (MWh_th / MWh_el)
                E_hp1  = sum(val(model.Qhp1[t])  for t in model.t) * dt
                E_hp2  = sum(val(model.Qhp2[t])  for t in model.t) * dt
                E_hp3  = sum(val(model.Qhp3[t])  for t in model.t) * dt
                E_hp4  = sum(val(model.Qhp4[t])  for t in model.t) * dt

                E_gtost_th = sum(val(model.GTOST_Power[t]) for t in model.t) * val(model.GTOST_th_eff) * dt
                E_p2h_th   = sum(val(model.P2H_Power[t])   for t in model.t) * val(model.P2H_eff)      * dt
                E_hkw_th   = sum(val(model.HKW_Power[t])   for t in model.t) * val(model.HKW_th_eff)    * dt
                E_bmhkw_th = sum(val(model.BMHKW_Power[t]) for t in model.t) * val(model.BMHKW_th_eff) * dt
                E_hws_th   = sum(val(model.HWS_Power[t])   for t in model.t) * val(model.HWS_th_eff)    * dt
                E_hww_th   = sum(val(model.HWW_Power[t])   for t in model.t) * val(model.HWW_th_eff)    * dt
                E_ava_th   = sum(val(model.AVA_Power[t])   for t in model.t) * val(model.AVA_th_eff)    * dt

                total_heat = (E_hp1 + E_hp2 + E_hp3 + E_hp4 +
                              E_bmhkw_th + E_gtost_th + E_p2h_th +
                              E_hkw_th + E_ava_th + E_hww_th + E_hws_th)

                E_gtost_el = sum(val(model.GTOST_Power[t]) for t in model.t) * val(model.GTOST_el_eff) * dt
                E_hkw_el   = sum(val(model.HKW_Power[t])   for t in model.t) * val(model.HKW_el_eff)   * dt
                E_bmhkw_el = sum(val(model.BMHKW_Power[t]) for t in model.t) * val(model.BMHKW_el_eff)* dt

                # Kosten/Erlöse
                total_buy_cost  = sum(dt * val(model.P_buy[t])  * val(model.buy_price[t])      for t in model.t)
                total_sell_rev  = sum(dt * val(model.P_sell[t]) * val(model.sell_price_eff[t]) for t in model.t)
                total_gas_cost  = sum(dt * val(model.Gasverbrauch[t]) * val(model.Gaspreis)    for t in model.t)

                hp_active_sum   = (val(model.HPNenn_active1) + val(model.HPNenn_active2) +
                                   val(model.HPNenn_active3) + val(model.HPNenn_active4))
                total_HP_cost =  ((val(model.HPNenn1)+val(model.HPNenn2)+val(model.HPNenn3)+val(model.HPNenn4)) * val(model.CapexHP)) / val(model.LebensdauerHP) * val(model.year_frac)
                total_HP_install_cost = hp_active_sum * val(model.Installationskosten) / val(model.LebensdauerHP) * val(model.year_frac)
                total_storage_cost    = val(model.CAPEXspeicher) * storage_cap / val(model.Lebensdauerspeicher) * val(model.year_frac)
                total_storage_inst    = val(model.storage_capacity_active) * val(model.Installationskosten) / val(model.Lebensdauerspeicher) * val(model.year_frac)
                total_power_capacity  = val(model.Leistungspreis) * val(model.max_stromverbrauch) * val(model.year_frac)

                total_profit = total_sell_rev - (total_buy_cost + total_gas_cost +
                                                 total_HP_cost +
                                                 total_HP_install_cost + total_storage_inst +
                                                 total_storage_cost + total_power_capacity)

                heat_price_eur_per_kwh = (total_profit / total_heat) / 1000 if total_heat > 0 else float('nan')

                results.append({
                    'CapexHP': capex_hp,
                    'CapexSpeicher': capex_speicher,
                    'Speicherkapazität_MWh': storage_cap,
                    'Gesamtkosten_EUR': total_cost,
                    'Kaufkosten_EUR': total_buy_cost,
                    'Verkaufserlöse_EUR': total_sell_rev,
                    'Gaskosten_EUR': total_gas_cost,
                    'HP_Install_EUR': total_HP_install_cost,
                    'Speicher_CAPEX_EUR': total_storage_cost,
                    'Speicher_Install_EUR': total_storage_inst,
                    'Leistungspreis_EUR': total_power_capacity,
                    'Gewinn_EUR': total_profit,
                    'E_HP1_MWh': E_hp1, 'E_HP2_MWh': E_hp2, 'E_HP3_MWh': E_hp3, 'E_HP4_MWh': E_hp4,
                    'E_GTOST_th_MWh': E_gtost_th, 'E_P2H_MWh': E_p2h_th, 'E_HKW_th_MWh': E_hkw_th,
                    'E_BMHKW_th_MWh': E_bmhkw_th, 'E_HWS_MWh': E_hws_th, 'E_HWW_MWh': E_hww_th, 'E_AVA_MWh': E_ava_th,
                    'E_GTOST_el_MWh': E_gtost_el, 'E_HKW_el_MWh': E_hkw_el, 'E_BMHKW_el_MWh': E_bmhkw_el,
                    'TotalHeat_MWh': total_heat,
                    'HeatPrice_EUR_per_kWh': heat_price_eur_per_kwh,
                    'HP1_kWth': val(model.HPNenn1), 'HP2_kWth': val(model.HPNenn2),
                    'HP3_kWth': val(model.HPNenn3), 'HP4_kWth': val(model.HPNenn4),
                })

                # >>> Timeseries des Szenarios in Sammel-Excel schreiben
                tag = f"HP{int(capex_hp)}_STO{int(capex_speicher)}"
                export_timeseries(model, DT_H,
                                  # start_ts="2025-01-01 00:00",  # optional echte Zeitachse
                                  writer=writer, sheet_prefix=tag,
                                  out_xlsx=None, out_csv=None)

            else:
                print(f"[WARN] Keine verwertbare Lösung für CapexHP={capex_hp}, CapexSpeicher={capex_speicher} "
                      f"(Status={result.solver.status}, TermCond={result.solver.termination_condition})")

# Aggregierte Kennzahlen separat ablegen
results_df = pd.DataFrame(results)
print(results_df)
results_df.to_csv("results.csv", index=False)
results_df.to_excel("results.xlsx", index=False)


Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2602149
Academic license 2602149 - for non-commercial use only - registered to lu___@eep.uni-stuttgart.de
Read LP format model from file C:\Users\LKR\AppData\Local\Temp\4\tmp1emsxhaa.pyomo.lp
Reading time = 2.22 seconds
x297853: 544581 rows, 297853 columns, 1384058 nonzeros
Set parameter TimeLimit to value 3600
Set parameter MIPGap to value 0.1
Set parameter LogToConsole to value 1
Set parameter Threads to value 30
Set parameter Heuristics to value 0.5
Set parameter Cuts to value 2
Set parameter Presolve to value 2
Set parameter Method to value 1
Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (win64 - Windows Server 2022.0 (20348.2))

CPU model: Intel(R) Xeon(R) Gold 5220 CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 34 physical cores, 66 logical processors, using up to 30 threads

Non-default parameters:
TimeLimit  3600
MIPGap  0.1
Method  1
Heuristics  0.5
Cuts  2
Presolve  2
Th

In [3]:
# ============================
# 7) Optimierung über Parameterbereiche (erweitert)
# ============================

# Definiere die Wertebereiche für CapexHP und CapexSpeicher
capex_hp_values = np.arange(100000, 1000001, 200000)  # 100000 bis 800000 in Schritten von 100000
capex_speicher_values = np.arange(1000, 10001, 2000)  # 1000 bis 8000 in Schritten von 1000

# Ergebnisse speichern
results = []

# Iteriere über alle Kombinationen der Parameter
for capex_hp in capex_hp_values:
    for capex_speicher in capex_speicher_values:
        # Setze die Parameter im Modell
        model.CapexHP.set_value(float(capex_hp))
        model.CAPEXspeicher.set_value(float(capex_speicher))

        
        # Lösen des Modells

       

        # Gurobi Parameter einstellen
        solver.options['TimeLimit'] = 3600  # 1 Stunde
        solver.options['MIPGap'] = 0.05      # 1% Lücke
        solver.options['LogToConsole'] = 1   # Log zur Konsole
        solver.options['Threads'] = 30         # Anzahl der Threads
        solver.options['OptimalityTol'] = 1e-6
        solver.options['Heuristics'] = 0.5    # Heuristiken aktivieren
        solver.options['Cuts'] = 2             # Schnitte aktivieren
        solver.options['Presolve'] = 2         # Starke Presolve-Option
        solver.options['Method'] = 1            # Dual Simplex verwenden
        #solver.options['LogFile'] = 'gurobi_log.txt'  # Log in Datei
        solver = SolverFactory('gurobi')
        result = solver.solve(model, tee=True)  # tee=True gibt die Solver-Ausgabe auf der Konsole aus

        # Überprüfe, ob das Modell erfolgreich gelöst wurde
        if result.solver.status == pyo.SolverStatus.ok and result.solver.termination_condition == pyo.TerminationCondition.optimal:
            # Werte abrufen
            storage_cap = pyo_val(model.storage_capacity)  
            total_cost = pyo_val(model.total_cost)  # Gesamtkosten
            HP1_power = pyo_val(model.HPNenn1)
            HP2_power = pyo_val(model.HPNenn2) 
            HP3_power = pyo_val(model.HPNenn3)
            HP4_power = pyo_val(model.HPNenn4)       

            # Summe der produzierten Wärme über den gesamten Zeitraum
            total_heat_produced_hp1 = pyo.sum(pyo_val(model.Qhp1[t]) for t in model.t)  # Produzierte Wärme Anlage 1
            total_heat_produced_hp2 = pyo.sum(pyo_val(model.Qhp2[t]) for t in model.t)  # Produzierte Wärme Anlage 2
            total_heat_produced_hp3 = pyo.sum(pyo_val(model.Qhp3[t]) for t in model.t)  # Produzierte Wärme Anlage 3
            total_heat_produced_hp4 = pyo.sum(pyo_val(model.Qhp4[t]) for t in model.t)  # Produzierte Wärme Anlage 4
            total_heat_produced_GTOST = pyo.sum(pyo_val(model.GTOST_Power[t]) * pyo_val(model.GTOST_th_eff) for t in model.t)  # Produzierte Wärme GTOST
            total_heat_produced_P2H = pyo.sum(pyo_val(model.P2H_Power[t]) * pyo_val(model.P2H_eff) for t in model.t)  # Produzierte Wärme P2H
            total_heat_produced_HKW = pyo.sum(pyo_val(model.HKW_Power[t]) * pyo_val(model.HKW_th_eff) for t in model.t)  # Produzierte Wärme HKW
            total_heat_produced_BMHKW = pyo.sum(pyo_val(model.BMHKW_Power[t]) * pyo_val(model.BMHKW_th_eff) for t in model.t)  # Produzierte Wärme BMHKW
            total_heat_produced_HWS = pyo.sum(pyo_val(model.HWS_Power[t]) * pyo_val(model.HWS_th_eff) for t in model.t)  # Produzierte Wärme HWS
            total_heat_produced_HWW = pyo.sum(pyo_val(model.HWW_Power[t]) * pyo_val(model.HWW_th_eff) for t in model.t)  # Produzierte Wärme HWW
            total_heat_produced_AVA = pyo.sum(pyo_val(model.AVA_Power[t]) * pyo_val(model.AVA_th_eff) for t in model.t)  # Produzierte Wärme AVA

            # Gesamte Wärmeproduktion
            total_heat = (total_heat_produced_hp1 + total_heat_produced_hp2 + total_heat_produced_hp3 + total_heat_produced_hp4 + 
                        total_heat_produced_BMHKW + total_heat_produced_GTOST + total_heat_produced_P2H +
                        total_heat_produced_HKW + total_heat_produced_AVA + total_heat_produced_HWW + 
                        total_heat_produced_HWS)      

            # Gesamte elektrische Energieproduktion
            total_electricity_produced_GTOST = pyo.sum(pyo_val(model.GTOST_Power[t]) * pyo_val(model.GTOST_el_eff) for t in model.t)
            total_electricity_produced_HKW = pyo.sum(pyo_val(model.HKW_Power[t]) * pyo_val(model.HKW_el_eff) for t in model.t)
            total_electricity_produced_BMHKW = pyo.sum(pyo_val(model.BMHKW_Power[t]) * pyo_val(model.BMHKW_el_eff) for t in model.t)   

            # Kosten aufsummieren
            total_buy_cost = pyo.sum(DT_H * pyo_val(model.P_buy[t]) * pyo_val(model.buy_price[t]) for t in model.t)  # Gesamte Kaufkosten
            total_sell_revenue = pyo.sum(DT_H * pyo_val(model.P_sell[t]) * pyo_val(model.sell_price_eff[t]) for t in model.t)  # Gesamte Verkaufserlöse
            total_gas_cost = pyo.sum(DT_H * pyo_val(model.Gasverbrauch[t]) * pyo_val(model.Gaspreis) for t in model.t)  # Gesamte Gaskosten
            total_HP_installation_cost = (pyo_val(model.HPNenn_active1) + pyo_val(model.HPNenn_active2) + 
                                        pyo_val(model.HPNenn_active3) + pyo_val(model.HPNenn_active4)) * pyo_val(model.Installationskosten) / pyo_val(model.LebensdauerHP) * pyo_val(model.year_frac)  # Installationskosten
            total_storage_cost = pyo_val(model.CAPEXspeicher) * storage_cap / pyo_val(model.Lebensdauerspeicher) * pyo_val(model.year_frac)  # Speicherkosten
            total_storage_installation_cost = (pyo_val(model.storage_capacity_active) * pyo_val(model.Installationskosten) / pyo_val(model.Lebensdauerspeicher) * pyo_val(model.year_frac))  # Installationskosten für Speicher
            total_leistungspreis = pyo_val(model.Leistungspreis) * pyo_val(model.max_stromverbrauch) * pyo_val(model.year_frac)  # Leistungspreis
            
            # Gewinne berechnen
            total_profit = total_sell_revenue - (total_buy_cost + total_gas_cost + total_HP_installation_cost + 
                                                total_storage_installation_cost + total_storage_cost + total_leistungspreis)
            Heat_Price = (total_profit / total_heat) / 1000  # Preis pro Wärmeeinheit

            # Ergebnisse speichern
            results.append({
                'CapexHP': capex_hp,
                'CapexSpeicher': capex_speicher,
                'Speicherkapazität': storage_cap,
                'Gesamtkosten': total_cost,
                'Kaufkosten': total_buy_cost,
                'Verkaufserlöse': total_sell_revenue,
                'Gaskosten': total_gas_cost,
                'Installationskosten': total_HP_installation_cost,
                'Speicherkosten': total_storage_cost,
                'Leistungspreis': total_leistungspreis,
                'Gewinn': total_profit,
                'TotalHeatProducedHP1': total_heat_produced_hp1,
                'TotalHeatProducedHP2': total_heat_produced_hp2,
                'TotalHeatProducedHP3': total_heat_produced_hp3,
                'TotalHeatProducedHP4': total_heat_produced_hp4,
                'TotalHeatProducedGTOST': total_heat_produced_GTOST,
                'TotalHeatProducedP2H': total_heat_produced_P2H,
                'TotalHeatProducedBMHKW': total_heat_produced_BMHKW,
                'TotalHeatProducedHWS': total_heat_produced_HWS,
                'TotalHeatProducedHWW': total_heat_produced_HWW,
                'TotalHeatProducedAVA': total_heat_produced_AVA,
                'Lebensdauer_HP': pyo_val(model.LebensdauerHP),
                'Speicher_Effizienz': pyo_val(model.storage_eff_charge),  
                'TotalElectricityProducedGTOST': total_electricity_produced_GTOST,
                'TotalElectricityProducedHKW': total_electricity_produced_HKW,
                'TotalElectricityProducedBMHKW': total_electricity_produced_BMHKW,
                'TotalHeat': total_heat,
                'HeatPrice': Heat_Price
            })
        else:
            print(f"Fehler bei der Lösung für CapexHP={capex_hp}, CapexSpeicher={capex_speicher}")

# Ergebnisse in DataFrame umwandeln für eine einfachere Analyse
results_df = pd.DataFrame(results)
print(results_df)

# Optional: Ergebnisse in eine CSV-Datei exportieren
results_df.to_csv("optimierungs_results_clean.csv", index=False)
# Ergebnisse in DataFrame umwandeln für eine einfachere Analyse
results_df.to_excel("optimierungs_results_clean.xlsx", index=False)

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2602149
Academic license 2602149 - for non-commercial use only - registered to lu___@eep.uni-stuttgart.de
Read LP format model from file C:\Users\LKR\AppData\Local\Temp\4\tmpt1w8n7sx.pyomo.lp
Reading time = 2.77 seconds
x297853: 544581 rows, 297853 columns, 1384058 nonzeros
Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (win64 - Windows Server 2022.0 (20348.2))

CPU model: Intel(R) Xeon(R) Gold 5220 CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 34 physical cores, 66 logical processors, using up to 32 threads

Academic license 2602149 - for non-commercial use only - registered to lu___@eep.uni-stuttgart.de
Optimize a model with 544581 rows, 297853 columns and 1384058 nonzeros
Model fingerprint: 0xff9b88ed
Model has 17520 quadratic constraints
Variable types: 210248 continuous, 87605 integer (87605 binary)
Coefficient statistics:
  Matrix range     [2e-01, 5e+04]
  QMatrix range    